# 🌿 WellAware AI - Healthcare LoRA Fine-Tuning Studio
**Project:** WellAware AI: Patient Wellness Campaign Visual Generator  
**Objective:** Fine-tune Stable Diffusion v1.5 using Low-Rank Adaptation (LoRA) on custom healthcare visuals on a free Google Colab T4 GPU.

### Step 1: Verify Colab GPU Acceleration

In [ ]:
import torch
print("=" * 60)
if torch.cuda.is_available():
    print(f"[SUCCESS] GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"[SUCCESS] Available VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[WARNING] Please switch to GPU: Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)

### Step 2: Install Required Libraries

In [ ]:
!pip install -q -U diffusers transformers accelerate safetensors torchvision

### Step 3: Create Dataset Directory & Starter Healthcare Samples
You can upload your own images into the `training_data/` folder, or use the starter set below.

In [ ]:
import os
from PIL import Image

os.makedirs("training_data", exist_ok=True)
os.makedirs("lora_models", exist_ok=True)

# Create sample placeholder data or place your custom images here
samples = [
    ("heart_health.png", "heart_health.txt", "wellaware style, 3d isometric illustration of glowing healthy heart, fruits, teal and coral tones"),
    ("diabetes_care.png", "diabetes_care.txt", "wellaware style, minimalist healthcare poster of diabetes nutrition plate, blue circle ribbon"),
    ("mindfulness.png", "mindfulness.txt", "wellaware style, peaceful mindfulness meditation silhouette, blooming lavender, soft sunlight"),
    ("hygiene.png", "hygiene.txt", "wellaware style, clean hands washing with foaming soap bubbles, crystal water droplets, turquoise tones")
]

for img_name, txt_name, caption in samples:
    img_path = os.path.join("training_data", img_name)
    txt_path = os.path.join("training_data", txt_name)
    if not os.path.exists(img_path):
        Image.new("RGB", (512, 512), color=(14, 116, 144)).save(img_path)
    with open(txt_path, "w") as f:
        f.write(caption)

print(f"[SUCCESS] {len(samples)} training pairs ready in 'training_data/'!")

### Step 4: Run LoRA Fine-Tuning

In [ ]:
# Execute the training loop
# (You can adjust --steps between 100 and 500)
!python -c "import prepare_dataset; prepare_dataset.prepare_starter_dataset()"
!python train_lora.py --steps 150 --lr 1e-4 --rank 8

### Step 5: Test Inference with Your Newly Trained LoRA Adapter

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch
from IPython.display import display

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)

# Load YOUR trained LoRA weights
lora_path = "lora_models/wellaware_healthcare_lora.safetensors"
if os.path.exists(lora_path):
    pipe.load_lora_weights(lora_path)
    print("[SUCCESS] Loaded custom trained LoRA weights!")

prompt = "wellaware style, 3d clay illustration of vibrant healthy heart, fresh organic apples, stethoscope, coral and teal tones, 8k"
image = pipe(prompt=prompt, num_inference_steps=20, guidance_scale=7.5).images[0]
image.save("lora_test_output.png")
display(image)